# 1-vazifa Ma’lumotlarni tayyorlash

* Kerakli kutubxonalarni import qilamiz.

* torchvision.datasets orqali FashionMNIST ma’lumotlar to‘plamini yuklab olamiz (train va test uchun alohida).

* Ma’lumotlarni ToTensor va Normalize((0.5,), (0.5,)) orqali o‘zgartiramiz.

* DataLoader yarating. batch_size qiymatini 32 qilib belgilaymiz.



In [1]:
#Kutubxonalar
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [2]:
torch.cuda.is_available()

True

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [5]:
#MNIST
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transforms
    )

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=transforms
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.5MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 203kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.76MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 17.5MB/s]


In [6]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

# 2-vazifa CNN Modelini qurish

* nn.Modul dan meros oluvchi yangi KiyimCNN klassini yaratamiz.

* nn.Sequential yordamida model arxitekturasini quyidagi parametarlar bilan quramiz: 1-blok: Conv2d (chiqish kanallari: 8), ReLU, MaxPool2d .

* Diqqat: Flatten qatlamidan keyingi birinchi Linear qatlamining kirish neyronlari sonini (in_features) o‘zingiz hisoblab toping.

* Modelni yaratib va uni device’ga o‘tkazamiz.

In [7]:
class KiyimCNN(nn.Module):
    def __init__(self):
        super(KiyimCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            #Flatten, linear, relu, linear
            nn.Flatten(),
            nn.Linear(in_features=7*7*16, out_features=128),
            nn.ReLU(),
            nn.Linear(in_features=128, out_features=10)

        )

    def forward(self, x):
        return self.features(x)

In [8]:
model = KiyimCNN().to(device)

#3-vazifa Modelni shug‘ullantirish va baholash

* Yo‘qotish funksiyasi (criterion) sifatida nn.CrossEntropyLoss'ni tanlaymiz.

* Optimizator (optimizer) sifatida optim.Adam dan foydalanib, lr (o‘rganish darajasi) qiymatini 0.002 qilamiz.

* Modelni 5 epoxa davomida shug‘ullantirib. Har bir epoxadan so‘ng o‘rtacha yo‘qotishni (total_loss) ekranga chiqaramiz.

* Shug‘ullantirish tugagach, test ma’lumotlari yordamida model aniqligini (accuracy) hisoblab va chop etamiz.



In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)

epox = 5

for epoch in range(epox):
    model.train()
    total_loss = 0.0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)

        #Forward - oldinga yuborish
        output = model(data)
        loss = criterion(output, target)

        #Backward - orqaga yuborish
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epox}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/5, Loss: 0.4458
Epoch 2/5, Loss: 0.3026
Epoch 3/5, Loss: 0.2587
Epoch 4/5, Loss: 0.2306
Epoch 5/5, Loss: 0.2088


In [13]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
  for data, target in test_loader:
    data, target = data.to(device), target.to(device)
    output = model(data)
    _, predicted = torch.max(output, 1)
    total += target.size(0)
    correct += (predicted == target).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 90.23%
